# MileStone 2 Project: Decomposing Paired-Rating Disagreement
## Part 0: Data Prepartion for Supervised and Unsupervised Learning
1. Combine training and holdout/test data file.
2. Merge the combined date with rater data.
3. QA and QC on the combined data file.

## Step 1: Combine both Train and Test Data Files

In [1]:
from pathlib import Path
import pandas as pd

# Folder containing the CSV files
DATA_PATH = Path(r'C:\Users\USEO9DA\0_MyStuff\M2\Data')
train = pd.read_csv(DATA_PATH / "ELLIPSE_Final_github_train.csv")
test = pd.read_csv(DATA_PATH / "ELLIPSE_Final_github_test.csv")

# Concatenate (stack rows)
combined = pd.concat([train, test], axis=0, ignore_index=True)

# Save combined file in the same folder
OUTPUT_PATH = DATA_PATH / "combined_train_test.csv"
combined.to_csv(OUTPUT_PATH, index=False)

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Combined shape:", combined.shape)
print("Saved to:", OUTPUT_PATH)

Train shape: (3911, 26)
Test shape: (2571, 26)
Combined shape: (6482, 26)
Saved to: C:\Users\USEO9DA\0_MyStuff\M2\Data\combined_train_test.csv


## Step 2: Merge Rater Data into the Combined Data
- Using TF DIF Method

In [2]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Defining file paths
DATA_PATH = Path(r'C:\Users\USEO9DA\0_MyStuff\M2\Data')
MAIN_FILE = DATA_PATH / "QA_Combined_Train_Test.csv"
RATER_FILE = DATA_PATH / "QA_ELL_Rater_Essay.csv"

# Deciding the parameters
TEXT_COL = "QA_Full_Text"
SIMILARITY_THRESHOLD = 0.80   # adjust as needed

# Creating helper function: cleaning text
def clean_text(text):
    if pd.isna(text):
        return ""    
    text = str(text).lower()
    text = text.replace("\n", " ").replace("\r", " ")
    text = re.sub(r"\s+", " ", text)        # collapse multiple spaces
    text = re.sub(r"[^\w\s]", "", text)     # remove punctuation
    return text.strip()

# Loading the files
df_main = pd.read_csv(MAIN_FILE)
df_rater = pd.read_csv(RATER_FILE)
print("Main file shape:", df_main.shape)
print("Rater file shape:", df_rater.shape)

if TEXT_COL not in df_main.columns:
    raise ValueError(f"Column '{TEXT_COL}' not found in {MAIN_FILE}")

if TEXT_COL not in df_rater.columns:
    raise ValueError(f"Column '{TEXT_COL}' not found in {RATER_FILE}")

# Keeping original row ids
df_main = df_main.copy()
df_rater = df_rater.copy()
df_main["main_row_id"] = np.arange(len(df_main))
df_rater["rater_row_id"] = np.arange(len(df_rater))

# Cleaning text
df_main["text_clean"] = df_main[TEXT_COL].apply(clean_text)
df_rater["text_clean"] = df_rater[TEXT_COL].apply(clean_text)

# Removing empty text rows
df_main = df_main[df_main["text_clean"] != ""].reset_index(drop=True)
df_rater = df_rater[df_rater["text_clean"] != ""].reset_index(drop=True)
print("Main shape after removing empty text:", df_main.shape)
print("Rater shape after removing empty text:", df_rater.shape)

# 7. Building TF-IDF vectors
    # Fitting on combined corpus so both files share the same vocabulary
all_texts = df_main["text_clean"].tolist() + df_rater["text_clean"].tolist()
vectorizer = TfidfVectorizer(stop_words="english",ngram_range=(1, 2),min_df=1)
tfidf_all = vectorizer.fit_transform(all_texts)
tfidf_main = tfidf_all[:len(df_main)]
tfidf_rater = tfidf_all[len(df_main):]
print("TF-IDF main shape:", tfidf_main.shape)
print("TF-IDF rater shape:", tfidf_rater.shape)

# Computing cosine similarity
sim_matrix = cosine_similarity(tfidf_main, tfidf_rater)
print("Similarity matrix shape:", sim_matrix.shape)

# Finding best match in rater.csv for each main row
best_match_indices = sim_matrix.argmax(axis=1)
best_match_scores = sim_matrix.max(axis=1)
df_main["matched_rater_index"] = best_match_indices
df_main["similarity_score"] = best_match_scores
print("\nSimilarity score summary:")
print(df_main["similarity_score"].describe())

# Splitting matched / unmatched
matched_df = df_main[df_main["similarity_score"] >= SIMILARITY_THRESHOLD].copy()
unmatched_df = df_main[df_main["similarity_score"] < SIMILARITY_THRESHOLD].copy()
print(f"\nMatched rows (>= {SIMILARITY_THRESHOLD}):", len(matched_df))
print(f"Unmatched rows (< {SIMILARITY_THRESHOLD}):", len(unmatched_df))

# Merging matched rows with rater data
df_rater_for_merge = df_rater.reset_index(drop=True).copy()
df_rater_for_merge["matched_rater_index"] = df_rater_for_merge.index
merged_df = matched_df.merge(df_rater_for_merge,on="matched_rater_index",how="left",suffixes=("_main", "_rater"))
print("Merged shape:", merged_df.shape)

OUTPUT_MATCHED_FILE = DATA_PATH / "merged_tfidf_match.csv"
OUTPUT_UNMATCHED_FILE = DATA_PATH / "unmatched_tfidf_match.csv"
merged_df .to_csv(OUTPUT_MATCHED_FILE, index=False)
unmatched_df.to_csv(OUTPUT_UNMATCHED_FILE, index=False)
print(f"\nSaved matched file: {OUTPUT_MATCHED_FILE}")
print(f"Saved unmatched file: {OUTPUT_UNMATCHED_FILE}")

# Previewing useful columns
preview_cols = ["main_row_id","rater_row_id","similarity_score","text_main","text_rater"]
existing_preview_cols = [col for col in preview_cols if col in merged_df.columns]
print("\nPreview of matched results:")
print(merged_df[existing_preview_cols].head(10))

Main file shape: (6482, 26)
Rater file shape: (8890, 20)
Main shape after removing empty text: (6482, 28)
Rater shape after removing empty text: (8888, 22)
TF-IDF main shape: (6482, 632303)
TF-IDF rater shape: (8888, 632303)
Similarity matrix shape: (6482, 8888)

Similarity score summary:
count    6482.000000
mean        0.985031
std         0.038125
min         0.628096
25%         0.989482
50%         1.000000
75%         1.000000
max         1.000000
Name: similarity_score, dtype: float64

Matched rows (>= 0.8): 6424
Unmatched rows (< 0.8): 58
Merged shape: (6424, 52)

Saved matched file: C:\Users\USEO9DA\0_MyStuff\M2\Data\merged_tfidf_match.csv
Saved unmatched file: C:\Users\USEO9DA\0_MyStuff\M2\Data\unmatched_tfidf_match.csv

Preview of matched results:
   main_row_id  rater_row_id  similarity_score
0            0          3233          1.000000
1            1          3310          0.926548
2            2          7487          1.000000
3            3          6865          1.000

## Step 3: Performing QA/QC on the Complete Merged Data
- Performing and Checking on the complete data
- Containng student's background variables, essay texts, and rater's scores, etc.
- Use score differences across seven dimensions

In [3]:
# Loading merged file
df = pd.read_csv(DATA_PATH / "merged_tfidf_match.csv")

# Reading in score categories
score_types = ["Overall","Cohesion","Syntax","Vocabulary","Phraseology","Grammar","Conventions"]

# Defining tolerance level (acceptable difference due to rounding)
TOLERANCE = 0.01

qa_results = []
for score in score_types:
    avg_col = score
    r1_col = score + "_1"
    r2_col = score + "_2"    
    df[score + "_calc_avg"] = (df[r1_col] + df[r2_col]) / 2     # Compute calculated average
    df[score + "_diff"] = df[avg_col] - df[score + "_calc_avg"]     # Compute difference
    df[score + "_flag"] = np.where(abs(df[score + "_diff"]) > TOLERANCE, 1, 0)     # Flag incorrect rows    
    # Store summary
    qa_results.append({
        "Score Type": score,
        "Num Errors": df[score + "_flag"].sum(),
        "Mean Diff": df[score + "_diff"].mean(),
        "Max Diff": df[score + "_diff"].abs().max()})

# Converting QA summary to DataFrame
qa_summary = pd.DataFrame(qa_results)
print("\n=== QA Summary ===")
print(qa_summary)

# Saving QA results to the Data folder
QA_MERGED_FILE = DATA_PATH / "merged_tfidf_match_with_QA.csv"
QA_SUMMARY_FILE = DATA_PATH / "QA_summary.csv"
df.to_csv(QA_MERGED_FILE, index=False)
qa_summary.to_csv(QA_SUMMARY_FILE, index=False)
print("\nQA files saved:")
print(" -", QA_MERGED_FILE)
print(" -", QA_SUMMARY_FILE)

df["QA_total_flags"] = df[[col for col in df.columns if "_flag" in col]].sum(axis=1)
# Creating QA flag (problem indicator)
df["QA_merge_problem"] = np.where(df["QA_total_flags"] >= 2, 1, 0)

# Final clean dataset (perfect matches only)
final_clean_df = df[df["QA_total_flags"] == 0].copy()
# Excluded dataset (any QA issue)
excluded_df = df[df["QA_total_flags"] != 0].copy()

# Saving final QA-cleaned results to the Data folder
FINAL_CLEAN_FILE = DATA_PATH / "final_clean_merged.csv"
EXCLUDED_FILE = DATA_PATH / "excluded_due_to_QA_flags.csv"

final_clean_df.to_csv(FINAL_CLEAN_FILE, index=False)
excluded_df.to_csv(EXCLUDED_FILE, index=False)
print("\nFinal QA files saved:")
print(" -", FINAL_CLEAN_FILE)
print(" -", EXCLUDED_FILE)

# 4. Summary report
print("========== QA Summary ==========")
print("Total rows:", len(df))
print("Final clean rows:", len(final_clean_df))
print("Excluded rows:", len(excluded_df))


=== QA Summary ===
    Score Type  Num Errors  Mean Diff  Max Diff
0      Overall          36   0.002024       1.5
1     Cohesion         115   0.002413       1.0
2       Syntax         105   0.000233       1.0
3   Vocabulary          60   0.000700       1.0
4  Phraseology         113   0.003580       1.5
5      Grammar         126   0.003113       1.0
6  Conventions         128   0.003892       1.5

QA files saved:
 - C:\Users\USEO9DA\0_MyStuff\M2\Data\merged_tfidf_match_with_QA.csv
 - C:\Users\USEO9DA\0_MyStuff\M2\Data\QA_summary.csv

Final QA files saved:
 - C:\Users\USEO9DA\0_MyStuff\M2\Data\final_clean_merged.csv
 - C:\Users\USEO9DA\0_MyStuff\M2\Data\excluded_due_to_QA_flags.csv
========== QA Summary ==========
Total rows: 6424
Final clean rows: 5840
Excluded rows: 584


# End Data Preparation Here